# 04. 비밀값·접근 권한·사이트 배포

| 항목 | 내용 |
|---|---|
| 목표 | 비밀값을 만들고, GitHub 에서 이미지·저장소를 받을 수 있게 한 뒤 ArgoCD 로 사이트를 배포 |
| 소요 시간 | 40분 |
| 시작 전 조건 | 03 완료(Funnel 주소), GitHub 토큰 1개 |
| 실행 위치 | 🖥 서버 콘솔 (절마다 다시 표시) |

> 대문자 대괄호 `[LIKE_THIS]` 는 **자기 환경의 값으로 바꿔 쓰는 자리표시자**입니다. 표는 [00_overview](00_overview.ipynb)에 있습니다.
> 이 노트북의 셀은 **문서**입니다. 설명대로 해당 위치(서버 콘솔 등)에서 직접 실행하세요. 명령에 따라 서버 설정이 바뀝니다.

## A. 네임스페이스와 비밀값 (🖥 서버 콘솔)

비밀값(앱 키, DB 비밀번호, SSH 인증서 발급용 키 등)을 **파일이나 저장소에 남기지 않고** 만들어 곧바로 클러스터에 넣습니다.

In [ ]:
!sudo k3s kubectl create namespace ssm
!bash ~/ssm/deploy/host/gen-secrets.sh | sudo k3s kubectl apply -f -
!sudo k3s kubectl -n ssm get secret ssm-secrets      # DATA 6

### 백업 (중요)
`SSM_SECRET_KEY` 를 잃으면 **모든 관리자의 인증 앱 등록과 서버 에이전트 연결이 무효**가 되어 다시 해야 합니다.
- 핵심 값 하나를 화면에 출력해 **비밀번호 관리자나 본인만 보는 메모**에 저장합니다. 채팅·이메일·사진은 피하세요(사진에 찍히면 키가 노출된 것이니 새로 만드세요).
- 서버 안에도 root 전용으로 한 부 둡니다.

In [ ]:
!sudo k3s kubectl -n ssm get secret ssm-secrets -o jsonpath='{.data.SSM_SECRET_KEY}' | base64 -d; echo
!sudo mkdir -p /root/ssm-backup && sudo sh -c 'umask 077; k3s kubectl -n ssm get secret ssm-secrets -o yaml > /root/ssm-backup/ssm-secrets.yaml'

## B. GitHub 토큰 만들기 (🌐 GitHub 웹)

서버가 ① 비공개 **이미지**(GHCR)를 받고 ② ArgoCD 가 비공개 **저장소**를 읽으려면 자격이 필요합니다.

1. GitHub → 우측 위 프로필 → **Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token (classic)**
2. 이름 `ssm-deploy`, 만료 90일, 권한 **`repo` 와 `read:packages` 둘 다** 체크 → 생성
3. 만들어진 값(`ghp_…`)을 **한 번만** 보여 주니 복사해 둡니다.

> `repo` 만 있으면 이미지가 `401`, `read:packages` 만 있으면 저장소가 `Write access … not granted` 로 실패합니다. **둘 다** 필요합니다.
> `repo` 는 내 모든 비공개 저장소에 대한 읽기·쓰기라 넓은 권한입니다. 사이트가 안정되면 ArgoCD 는 **읽기 전용 Deploy key** 로 바꾸고 이 토큰은 폐기하는 것을 권합니다.

## C. 토큰을 서버에 안전하게 넣기

토큰을 명령줄에 직접 쓰면 명령 기록에 남고 오타도 찾기 어렵습니다. `read -r` 로 **빈 줄에 입력**받습니다.

1. `read -r T1` 을 치면 프롬프트가 사라지고 **빈 줄에서 커서만 깜빡입니다**(정상). 그 줄에 토큰만 치고 Enter.
2. 프롬프트가 돌아오면 글자 수와 권한을 확인합니다. 글자 수는 **40**, 권한 줄에 **`read:packages, repo`** 가 있어야 합니다.
3. `read` 를 친 뒤 같은 줄에 다른 명령을 치면 그 글자가 토큰으로 들어갑니다(흔한 실수).

In [ ]:
!read -r T1
!echo ${#T1}                                    # 40
!curl -sI -H "Authorization: token $T1" https://api.github.com/user | grep -i "^x-oauth-scopes:"
!curl -s -o /dev/null -w '%{http_code}\n' -H "Authorization: token $T1" https://api.github.com/user     # 200

## D. 이미지·저장소 자격 만들기

- `ghcr-pull`: 파드가 비공개 이미지를 받는 자격. **이 시크릿이 없거나 값이 비어 있으면** 파드에 `failed to fetch anonymous token … 401` 이 나옵니다(익명으로 접속했다는 뜻).
- `repo-ssm`: ArgoCD 가 저장소를 읽는 자격.
- 토큰이 GHCR 에서 통하는지 **직접 받아 보는** 확인도 있습니다(성공하면 이미지가 서버에 미리 받아집니다).

In [ ]:
!sudo k3s crictl pull --creds [GITHUB_USER]:$T1 ghcr.io/[github_user_소문자]/ssm-backend:[IMAGE_TAG]
!sudo k3s kubectl -n ssm create secret docker-registry ghcr-pull --docker-server=ghcr.io --docker-username=[GITHUB_USER] --docker-password="$T1"
!sudo k3s kubectl apply -f ~/ssm/deploy/argocd/project.yaml
!sudo k3s kubectl -n argocd create secret generic repo-ssm --from-literal=type=git --from-literal=url=https://github.com/[GITHUB_USER]/[REPO].git --from-literal=username=[GITHUB_USER] --from-literal=password="$T1"
!sudo k3s kubectl -n argocd label secret repo-ssm argocd.argoproj.io/secret-type=repository
!unset T1

## E. 사이트 배포 (ArgoCD 앱 만들기)

저장소의 `deploy/prod/` 에 운영 값(`values-prod.yaml`: 주소·사설 IP)과 이미지 태그(`image-tag.yaml`)가 있습니다.
이미지 태그는 GitHub Actions 의 `release` 워크플로가 이미지를 GHCR 에 올린 뒤 **자동으로** 갱신합니다.

In [ ]:
!sudo k3s kubectl apply -f ~/ssm/deploy/argocd/application.yaml
# 1~3분 뒤
!sudo k3s kubectl -n argocd get application ssm        # Synced / Healthy
!sudo k3s kubectl -n ssm get pods                      # 모두 Running (처음엔 이미지를 받느라 시간이 걸림)
!sudo k3s kubectl -n ssm exec deploy/ssm-api -- python -m alembic current      # 0004 (head) 처럼 head 가 보임

파드 상태를 **실시간으로** 보려면 (끝내기 `Ctrl+C`):

In [ ]:
!sudo watch -n 2 k3s kubectl -n ssm get pods

## 배포가 막혔을 때 (자세한 사례는 08)
| 증상 | 원인·조치 |
|---|---|
| 앱이 `Unknown`, `Write access to repository not granted` | 토큰에 `repo` 권한이 없음 → 토큰 권한 추가 후 `refresh=hard` |
| 파드 `ImagePullBackOff`, `anonymous token … 401` | `ghcr-pull` 이 없거나 값이 비었음 → 토큰으로 다시 만들기 |
| 앱이 `OutOfSync` 로 멈춤 | 첫 동기화가 실패한 기록이 남음 → 앱 정의 삭제 후 다시 `apply` |
| 새 설정이 안 올라옴 | `sudo k3s kubectl -n argocd annotate application ssm argocd.argoproj.io/refresh=hard --overwrite` |

앱 정의를 지우고 다시 만들어도 **파드·DB 데이터는 지워지지 않습니다**(앱 정의에 삭제 시 리소스를 함께 지우는 설정이 없음).

## ✅ 완료 확인

- [ ] `ssm-secrets` 의 `SSM_SECRET_KEY` 를 서버 밖에 백업했다
- [ ] 토큰이 `read:packages, repo` 권한이고 글자 수 40, 상태 코드 200
- [ ] `ghcr-pull`, `repo-ssm` 시크릿이 있다
- [ ] 앱이 Synced/Healthy, 파드가 모두 Running
- [ ] 브라우저에서 사이트 로그인 화면이 열린다